In [83]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [84]:
import pandas as pd
train = pd.read_json('/content/drive/MyDrive/DL-TH4/data/small-train.json')
dev = pd.read_json('/content/drive/MyDrive/DL-TH4/data/small-dev.json')
test = pd.read_json('/content/drive/MyDrive/DL-TH4/data/small-test.json')

In [85]:
%%writefile /content/drive/MyDrive/DL-TH4/Vocab.py
import os
import json
import torch
import re

class Vocab:
    def __init__(self, src_language: str, tgt_language: str):
        self.src_language = src_language
        self.tgt_language = tgt_language
        self.initialize_special_tokens()

    def initialize_special_tokens(self):
        self.pad_token = "<pad>"
        self.bos_token = "<bos>"
        self.eos_token = "<eos>"
        self.unk_token = "<unk>"
        self.specials = [self.pad_token, self.bos_token, self.eos_token, self.unk_token]
        self.pad_idx = 0
        self.bos_idx = 1
        self.eos_idx = 2
        self.unk_idx = 3

    def preprocess_sentence(self, sentence: str):
        sentence = sentence.lower()
        return re.findall(r"\w+", sentence)

    def make_vocab(self, path: str):
        src_words = set()
        tgt_words = set()
        for file in os.listdir(path):
            data = json.load(open(os.path.join(path, file), encoding="utf-8"))
            for item in data:
                src_words.update(self.preprocess_sentence(item[self.src_language]))
                tgt_words.update(self.preprocess_sentence(item[self.tgt_language]))
        src_itos = self.specials + list(src_words)
        tgt_itos = self.specials + list(tgt_words)
        self.src_itos = {i: tok for i, tok in enumerate(src_itos)}
        self.src_stoi = {tok: i for i, tok in enumerate(src_itos)}
        self.tgt_itos = {i: tok for i, tok in enumerate(tgt_itos)}
        self.tgt_stoi = {tok: i for i, tok in enumerate(tgt_itos)}

    def encode_sentence(self, sentence: str, language: str):
        tokens = self.preprocess_sentence(sentence)
        stoi = self.src_stoi if language == self.src_language else self.tgt_stoi
        vec = [self.bos_idx] + [stoi.get(tok, self.unk_idx) for tok in tokens] + [self.eos_idx]
        return torch.tensor(vec, dtype=torch.long)

    def decode_sentence(self, vec: torch.Tensor, language: str):
        ids = vec.tolist()
        itos = self.src_itos if language == self.src_language else self.tgt_itos
        words = []
        for idx in ids:
            if idx == self.eos_idx:
                break
            words.append(itos[idx])
        return " ".join(words)

    def total_src_tokens(self):
        return len(self.src_itos)

    def total_tgt_tokens(self):
        return len(self.tgt_itos)

Overwriting /content/drive/MyDrive/DL-TH4/Vocab.py


In [86]:
%%writefile /content/drive/MyDrive/DL-TH4/Dataset.py
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
import json

class PhoMTDataset(Dataset):
    def __init__(self, path, src_vocab, tgt_vocab):
        with open(path, encoding="utf-8") as f:
            self.data = json.load(f)
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        # Encode sẵn để DataLoader nhanh
        self.data = [{"en": src_vocab.encode_sentence(item["english"], "english"),
                      "vi": tgt_vocab.encode_sentence(item["vietnamese"], "vietnamese")}
                     for item in self.data]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    en = [item["en"] for item in batch]
    vi = [item["vi"] for item in batch]
    en = pad_sequence(en, batch_first=True, padding_value=0)
    vi = pad_sequence(vi, batch_first=True, padding_value=0)
    return en, vi

Overwriting /content/drive/MyDrive/DL-TH4/Dataset.py


# Bài 1

In [87]:
%%writefile /content/drive/MyDrive/DL-TH4/LSTM.py
import torch
import torch.nn as nn

class Seq2SeqLSTM(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, pad_idx):
        super().__init__()
        self.hidden_size = 256
        self.num_layers = 3
        self.src_embedding = nn.Embedding(src_vocab_size, 256, padding_idx=pad_idx)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, 256, padding_idx=pad_idx)
        self.encoder = nn.LSTM(256, 256, 3, batch_first=True)
        self.decoder = nn.LSTM(256, 256, 3, batch_first=True)
        self.fc = nn.Linear(256, tgt_vocab_size)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=pad_idx)

    def forward(self, src, tgt):
        embedded_src = self.src_embedding(src)
        _, (h, c) = self.encoder(embedded_src)
        embedded_tgt = self.tgt_embedding(tgt[:, :-1])
        outputs, _ = self.decoder(embedded_tgt, (h, c))
        logits = self.fc(outputs)
        return logits

    def compute_loss(self, logits, tgt):
        return self.loss_fn(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))


Overwriting /content/drive/MyDrive/DL-TH4/LSTM.py


In [88]:
!pip install rouge-score

In [89]:
%%writefile /content/drive/MyDrive/DL-TH4/train.py
import torch
from torch.utils.data import DataLoader
from Dataset import PhoMTDataset, collate_fn
from Vocab import Vocab
from LSTM import Seq2SeqLSTM
from tqdm import tqdm
import pandas as pd
import json
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = '/content/drive/MyDrive/DL-TH4/data/'

# Vocab
src_vocab = Vocab("english", "vietnamese")
tgt_vocab = Vocab("english", "vietnamese")
src_vocab.make_vocab(data_path)
tgt_vocab.make_vocab(data_path)

# Dataset
train_dataset = PhoMTDataset(data_path + 'small-train.json', src_vocab, tgt_vocab)
dev_dataset   = PhoMTDataset(data_path + 'small-dev.json', src_vocab, tgt_vocab)
test_dataset  = PhoMTDataset(data_path + 'small-test.json', src_vocab, tgt_vocab)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# Model
model = Seq2SeqLSTM(src_vocab.total_src_tokens(), tgt_vocab.total_tgt_tokens(), src_vocab.pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train
num_epochs = 5
for epoch in range(1, num_epochs+1):
    model.train()
    total_loss = 0
    for en_batch, vi_batch in tqdm(train_loader):
        en_batch, vi_batch = en_batch.to(device), vi_batch.to(device)
        optimizer.zero_grad()
        logits = model(en_batch, vi_batch)
        loss = model.compute_loss(logits, vi_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/{num_epochs}, Avg Loss: {total_loss/len(train_loader):.4f}")

# Save model
model_path = '/content/drive/MyDrive/DL-TH4/seq2seq_20k_train.pth'
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

# Evaluate ROUGE-L
def evaluate(loader, dataset_name):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = []
    with torch.no_grad():
        for en_batch, vi_batch in loader:
            en_batch, vi_batch = en_batch.to(device), vi_batch.to(device)
            logits = model(en_batch, vi_batch)
            pred_ids = torch.argmax(logits, dim=-1)
            for pred, ref in zip(pred_ids, vi_batch):
                pred_sent = tgt_vocab.decode_sentence(pred, "vietnamese")
                ref_sent  = tgt_vocab.decode_sentence(ref, "vietnamese")
                scores.append(scorer.score(ref_sent, pred_sent)['rougeL'].fmeasure)
    avg_score = sum(scores)/len(scores)
    print(f"ROUGE-L trung bình trên {dataset_name}: {avg_score:.4f}")
    return avg_score

evaluate(dev_loader, "Dev")
evaluate(test_loader, "Test")

Overwriting /content/drive/MyDrive/DL-TH4/train.py


In [90]:
!python /content/drive/MyDrive/DL-TH4/train.py

100% 2500/2500 [00:38<00:00, 64.45it/s]
Epoch 1/5, Avg Loss: 5.7489
100% 2500/2500 [00:38<00:00, 65.47it/s]
Epoch 2/5, Avg Loss: 4.8878
100% 2500/2500 [00:39<00:00, 63.75it/s]
Epoch 3/5, Avg Loss: 4.4777
100% 2500/2500 [00:38<00:00, 64.81it/s]
Epoch 4/5, Avg Loss: 4.2111
100% 2500/2500 [00:38<00:00, 64.89it/s]
Epoch 5/5, Avg Loss: 4.0070
Model saved to /content/drive/MyDrive/DL-TH4/seq2seq_20k_train.pth
ROUGE-L trung bình trên Dev: 0.3501
ROUGE-L trung bình trên Test: 0.3621


# Bài 2

In [91]:
%%writefile /content/drive/MyDrive/DL-TH4/LSTM_Bahdanau.py
import torch
from torch.utils.data import DataLoader
from Dataset import PhoMTDataset, collate_fn
from Vocab import Vocab
from tqdm import tqdm
from rouge_score import rouge_scorer
import torch.nn as nn

class Seq2SeqBahdanau(nn.Module):
    def __init__(self, vocab: Vocab, d_model=256, n_encoder=2, n_decoder=2, dropout=0.1):
        super().__init__()
        self.vocab = vocab
        self.d_model = d_model
        self.n_encoder = n_encoder
        self.n_decoder = n_decoder

        self.src_embedding = nn.Embedding(vocab.total_src_tokens(), d_model, padding_idx=vocab.pad_idx)
        self.tgt_embedding = nn.Embedding(vocab.total_tgt_tokens(), d_model, padding_idx=vocab.pad_idx)

        self.encoder = nn.LSTM(d_model, d_model, n_encoder, batch_first=True, bidirectional=True, dropout=dropout)
        self.decoder = nn.LSTM(d_model + 2*d_model, 2*d_model, n_decoder, batch_first=True)

        # Bahdanau attention
        self.attn = nn.Linear(2*d_model + 2*d_model, 2*d_model)
        self.v = nn.Linear(2*d_model, 1)

        self.fc_out = nn.Linear(2*d_model, vocab.total_tgt_tokens())
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx)

    def forward(self, src, tgt):
        bs, src_len = src.size()
        bs, tgt_len = tgt.size()
        device = src.device

        # encoder
        enc_emb = self.src_embedding(src)
        enc_outputs, (h, c) = self.encoder(enc_emb)  # enc_outputs: (bs, src_len, 2*d_model)

        # decoder initial state
        dec_h = torch.zeros(self.n_decoder, bs, 2*self.d_model).to(device)
        dec_c = torch.zeros(self.n_decoder, bs, 2*self.d_model).to(device)

        logits = []
        tgt_emb = self.tgt_embedding(tgt[:, :-1])

        for t in range(tgt_emb.size(1)):
            y_t = tgt_emb[:, t, :].unsqueeze(1)  # (bs,1,d_model)

            # compute attention
            dec_h_last = dec_h[-1].unsqueeze(1).repeat(1, src_len, 1)  # (bs, src_len, 2*d_model)
            energy = torch.tanh(self.attn(torch.cat([dec_h_last, enc_outputs], dim=-1)))  # (bs, src_len, 2*d_model)
            attn_weights = torch.softmax(self.v(energy), dim=1)  # (bs, src_len, 1)
            context = torch.sum(attn_weights * enc_outputs, dim=1, keepdim=True)  # (bs,1,2*d_model)

            dec_input = torch.cat([y_t, context], dim=-1)
            _, (dec_h, dec_c) = self.decoder(dec_input, (dec_h, dec_c))

            logit = self.fc_out(dec_h[-1])
            logits.append(logit.unsqueeze(1))

        logits = torch.cat(logits, dim=1)
        loss = self.loss_fn(logits.reshape(-1, logits.size(-1)), tgt[:,1:].reshape(-1))
        return loss, logits

Overwriting /content/drive/MyDrive/DL-TH4/LSTM_Bahdanau.py


In [92]:
%%writefile /content/drive/MyDrive/DL-TH4/train_bahdanau.py
import torch
from torch.utils.data import DataLoader
from Dataset import PhoMTDataset, collate_fn
from Vocab import Vocab
from LSTM_Bahdanau import Seq2SeqBahdanau
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = '/content/drive/MyDrive/DL-TH4/data/'

# vocab
src_vocab = Vocab("english","vietnamese")
tgt_vocab = Vocab("english","vietnamese")
src_vocab.make_vocab(data_path)
tgt_vocab.make_vocab(data_path)

# dataset
train_dataset = PhoMTDataset(data_path + 'small-train.json', src_vocab, tgt_vocab)
dev_dataset   = PhoMTDataset(data_path + 'small-dev.json', src_vocab, tgt_vocab)
test_dataset  = PhoMTDataset(data_path + 'small-test.json', src_vocab, tgt_vocab)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# model
model = Seq2SeqBahdanau(vocab=src_vocab).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# train
num_epochs = 5
for epoch in range(1, num_epochs+1):
    model.train()
    total_loss = 0
    for src_batch, tgt_batch in tqdm(train_loader):
        src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)
        optimizer.zero_grad()
        loss, _ = model(src_batch, tgt_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/{num_epochs}, Avg Loss: {total_loss/len(train_loader):.4f}")

# save
torch.save(model.state_dict(), '/content/drive/MyDrive/DL-TH4/seq2seq_bahdanau.pth')

# evaluate ROUGE-L
def evaluate(loader, name):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = []
    with torch.no_grad():
        for src_batch, tgt_batch in loader:
            src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)
            _, logits = model(src_batch, tgt_batch)
            pred_ids = torch.argmax(logits, dim=-1)
            for pred, ref in zip(pred_ids, tgt_batch):
                pred_sent = tgt_vocab.decode_sentence(pred, "vietnamese")
                ref_sent  = tgt_vocab.decode_sentence(ref, "vietnamese")
                scores.append(scorer.score(ref_sent, pred_sent)['rougeL'].fmeasure)
    avg = sum(scores)/len(scores)
    print(f"ROUGE-L trung bình trên {name}: {avg:.4f}")
    return avg

evaluate(dev_loader, "Dev")
evaluate(test_loader, "Test")

Overwriting /content/drive/MyDrive/DL-TH4/train_bahdanau.py


In [93]:
!python /content/drive/MyDrive/DL-TH4/train_bahdanau.py

100% 2500/2500 [05:37<00:00,  7.40it/s]
Epoch 1/5, Avg Loss: 5.9465
100% 2500/2500 [05:35<00:00,  7.45it/s]
Epoch 2/5, Avg Loss: 4.9118
100% 2500/2500 [05:33<00:00,  7.50it/s]
Epoch 3/5, Avg Loss: 4.3269
100% 2500/2500 [05:32<00:00,  7.51it/s]
Epoch 4/5, Avg Loss: 3.9424
100% 2500/2500 [05:32<00:00,  7.52it/s]
Epoch 5/5, Avg Loss: 3.6324
ROUGE-L trung bình trên Dev: 0.3766
ROUGE-L trung bình trên Test: 0.3925


# Bài 3

In [94]:
%%writefile /content/drive/MyDrive/DL-TH4/LSTM_luong.py
import torch
from torch.utils.data import DataLoader
from Dataset import PhoMTDataset, collate_fn
from Vocab import Vocab
from tqdm import tqdm
import torch.nn as nn
from rouge_score import rouge_scorer

class Seq2SeqLuong(nn.Module):
    def __init__(self, vocab: Vocab, d_model=256, n_encoder=2, n_decoder=2, dropout=0.1):
        super().__init__()
        self.vocab = vocab
        self.d_model = d_model
        self.n_encoder = n_encoder
        self.n_decoder = n_decoder

        self.src_embedding = nn.Embedding(vocab.total_src_tokens(), d_model, padding_idx=vocab.pad_idx)
        self.tgt_embedding = nn.Embedding(vocab.total_tgt_tokens(), d_model, padding_idx=vocab.pad_idx)

        self.encoder = nn.LSTM(d_model, d_model, n_encoder, batch_first=True, bidirectional=True, dropout=dropout)
        self.decoder = nn.LSTM(d_model + 2*d_model, 2*d_model, n_decoder, batch_first=True)

        # Luong attention
        self.attn = nn.Linear(2*d_model, 2*d_model)

        self.fc_out = nn.Linear(2*d_model, vocab.total_tgt_tokens())
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx)

    def forward(self, src, tgt):
        bs, src_len = src.size()
        bs, tgt_len = tgt.size()
        device = src.device

        # encoder
        enc_emb = self.src_embedding(src)
        enc_outputs, (h, c) = self.encoder(enc_emb)

        # decoder initial state
        dec_h = torch.zeros(self.n_decoder, bs, 2*self.d_model).to(device)
        dec_c = torch.zeros(self.n_decoder, bs, 2*self.d_model).to(device)

        logits = []
        tgt_emb = self.tgt_embedding(tgt[:, :-1])

        for t in range(tgt_emb.size(1)):
            y_t = tgt_emb[:, t, :].unsqueeze(1)

            # compute attention
            dec_h_last = dec_h[-1].unsqueeze(1)
            score = torch.bmm(dec_h_last, enc_outputs.transpose(1,2))
            attn_weights = torch.softmax(score, dim=-1)
            context = torch.bmm(attn_weights, enc_outputs)

            dec_input = torch.cat([y_t, context], dim=-1)
            _, (dec_h, dec_c) = self.decoder(dec_input, (dec_h, dec_c))

            logit = self.fc_out(dec_h[-1])
            logits.append(logit.unsqueeze(1))

        logits = torch.cat(logits, dim=1)
        loss = self.loss_fn(logits.reshape(-1, logits.size(-1)), tgt[:,1:].reshape(-1))
        return loss, logits

Overwriting /content/drive/MyDrive/DL-TH4/LSTM_luong.py


In [97]:
%%writefile "/content/drive/MyDrive/DL-TH4/train_luong.py"
import torch
from torch.utils.data import DataLoader
from Dataset import PhoMTDataset, collate_fn
from Vocab import Vocab
from LSTM_luong import Seq2SeqLuong
from tqdm import tqdm
from rouge_score import rouge_scorer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = '/content/drive/MyDrive/DL-TH4/data/'

# vocab
src_vocab = Vocab("english","vietnamese")
tgt_vocab = Vocab("english","vietnamese")
src_vocab.make_vocab(data_path)
tgt_vocab.make_vocab(data_path)

# dataset
train_dataset = PhoMTDataset(data_path + 'small-train.json', src_vocab, tgt_vocab)
dev_dataset   = PhoMTDataset(data_path + 'small-dev.json', src_vocab, tgt_vocab)
test_dataset  = PhoMTDataset(data_path + 'small-test.json', src_vocab, tgt_vocab)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# model
model = Seq2SeqLuong(vocab=src_vocab).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# train
num_epochs = 5
for epoch in range(1, num_epochs+1):
    model.train()
    total_loss = 0
    for src_batch, tgt_batch in tqdm(train_loader):
        src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)
        optimizer.zero_grad()
        loss, _ = model(src_batch, tgt_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/{num_epochs}, Avg Loss: {total_loss/len(train_loader):.4f}")

# save
torch.save(model.state_dict(), '/content/drive/MyDrive/DL-TH4/seq2seq_luong.pth')

# evaluate ROUGE-L
def evaluate(loader, name):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = []
    with torch.no_grad():
        for src_batch, tgt_batch in loader:
            src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)
            _, logits = model(src_batch, tgt_batch)
            pred_ids = torch.argmax(logits, dim=-1)
            for pred, ref in zip(pred_ids, tgt_batch):
                pred_sent = tgt_vocab.decode_sentence(pred, "vietnamese")
                ref_sent  = tgt_vocab.decode_sentence(ref, "vietnamese")
                scores.append(scorer.score(ref_sent, pred_sent)['rougeL'].fmeasure)
    avg = sum(scores)/len(scores)
    print(f"ROUGE-L trung bình trên {name}: {avg:.4f}")
    return avg

evaluate(dev_loader, "Dev")
evaluate(test_loader, "Test")

Overwriting /content/drive/MyDrive/DL-TH4/train_luong.py


In [98]:
!python /content/drive/MyDrive/DL-TH4/train_luong.py

100% 2500/2500 [04:28<00:00,  9.30it/s]
Epoch 1/5, Avg Loss: 5.7333
100% 2500/2500 [04:26<00:00,  9.37it/s]
Epoch 2/5, Avg Loss: 4.7362
100% 2500/2500 [04:26<00:00,  9.37it/s]
Epoch 3/5, Avg Loss: 4.2791
100% 2500/2500 [04:31<00:00,  9.21it/s]
Epoch 4/5, Avg Loss: 3.9638
100% 2500/2500 [04:27<00:00,  9.35it/s]
Epoch 5/5, Avg Loss: 3.6965
ROUGE-L trung bình trên Dev: 0.3718
ROUGE-L trung bình trên Test: 0.3855
